# Assigned is not always delivered

**A guided investigation · Chapter 3 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-mdve_clean.ipynb)

> **The question**  
> When officers were assigned a response, how often was that response delivered?

In Minneapolis in 1981–1982, researchers studied police responses to eligible misdemeanor domestic assault incidents. Officers were assigned one of three responses: arrest, advice, or separating the parties. The wider study asked whether these responses affected repeat violence. In practice, an assigned response was not always the response delivered. This teaching extract contains 314 records with only those two categorical fields; the original study involved 330 cases, and the extract does not contain subsequent violence outcomes. We can therefore investigate how assignment translated into action, but cannot estimate whether arrest reduced repeat violence from this file. A count chart and a cross-tabulation make the distinction visible. It is a concrete introduction to noncompliance: even when an assignment is randomized, the treatment actually received may reflect additional decisions and circumstances.

### By the end, you will be able to…

- Trace the difference between assigned and delivered responses, then identify the evidence still missing.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

Would comparing people actually arrested with people not arrested preserve the original random assignment?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Police response to an eligible misdemeanor domestic assault call (330 cases in source study).  
**Stored size:** 314 rows × 2 columns.

| Column | What it records |
|---|---|
| `assigned` | Police response assigned by the experiment: Arrest, Advise, or Separate. |
| `delivered` | Police response actually delivered; compare its category label with assigned. |

This extract has no recidivism outcome. Its 314 records must not be described as all 330 study cases. Use respectful language: these records concern serious harm to people.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch3/mdve_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['assigned', 'delivered']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['assigned', 'delivered']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'census': 'Census year',
 'group': 'State group',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state_name': 'Simulated state'}

## Visual investigation 1 · Count the assigned responses

**Predict:** Were all three assignments equally common in the stored extract?

value_counts counts category labels. reset_index gives the counts column names so we can plot them. These are cases in this extract, not the entire source study.

In [ ]:
counts = df["assigned"].value_counts().rename_axis("assigned").reset_index(name="cases")
fig = px.bar(counts, x="assigned", y="cases", color="assigned", labels={"assigned": "Assigned response", "cases": "Cases in extract"}, title="The starting assignment")
fig.show()

**Explore the chart:** Hover over each bar. Is equal random-assignment probability the same thing as exactly equal realized counts?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Finite samples can have unequal counts. Counts alone cannot show whether assignment was implemented correctly or whether omitted cases differ.

</details>

## Visual investigation 2 · Follow assignment into action

**Predict:** For each assigned response, what was actually delivered?

crosstab builds a table of category combinations. normalize="index" divides each row by its own total, so each assigned-response row adds to 100%.

In [ ]:
shares = pd.crosstab(df["assigned"], df["delivered"], normalize="index") * 100
fig = px.imshow(shares, text_auto=".1f", zmin=0, zmax=100, color_continuous_scale="Blues", labels={"x": "Delivered response", "y": "Assigned response", "color": "Within assignment (%)"}, title="Where assignment and action diverge")
fig.show()

**Explore the chart:** Hover over a diagonal cell (matching responses), then an off-diagonal cell. Explain both denominators.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Off-diagonal cells show noncompliance. A comparison based on delivered responses can therefore sort cases differently from the randomized assignment.

</details>

## Visual investigation 3 · Summarize the mismatch

**Predict:** Which assignments were most often delivered as assigned?

== compares two values and returns True or False. The mean of these values is a proportion because True counts as 1 and False as 0.

In [ ]:
compliance = df.assign(matched=df["assigned"] == df["delivered"])
matched = compliance.groupby("assigned", as_index=False)["matched"].mean()
matched["percent"] = 100 * matched["matched"]
fig = px.bar(matched, x="assigned", y="percent", labels={"assigned": "Assigned response", "percent": "Delivered as assigned (%)"}, title="Compliance depends on the assignment")
fig.update_yaxes(range=[0, 100])
fig.show()

**Explore the chart:** Identify the smallest match rate. Describe the result without claiming a cause of noncompliance.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

You have measured assignment fidelity, not the effect on violence. The absence of an outcome variable sets a firm boundary on this investigation.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Change normalize="index" to normalize="columns" in the heatmap. Explain the new denominator before interpreting it.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Plot the distribution of delivered responses only among cases assigned to Separate.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Filter with df[df["assigned"] == "Separate"].

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Count delivered values in the filtered table, then use px.bar.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>separate = df[df[&quot;assigned&quot;] == &quot;Separate&quot;]
counts = separate[&quot;delivered&quot;].value_counts().rename_axis(&quot;delivered&quot;).reset_index(name=&quot;cases&quot;)
fig = px.bar(counts, x=&quot;delivered&quot;, y=&quot;cases&quot;, labels={&quot;delivered&quot;: &quot;Delivered response&quot;, &quot;cases&quot;: &quot;Cases assigned Separate&quot;})
fig.show()</code></pre>

The bars follow one assignment group into its delivered responses. They do not show which response worked best.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**Why might assigned response be a more credible starting point than delivered response when investigating an arrest effect?**

[Continue with Chapter 3: Instrumental Variables](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/03-instrumental-variables.html#the-minneapolis-domestic-violence-experiment) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch3-mdve-clean-csv)

### Source and preparation notes

[Sherman and Berk (1984), The Minneapolis Domestic Violence Experiment](https://www.ojp.gov/library/publications/minneapolis-domestic-violence-experiment)

The 1981–82 field experiment randomly assigned arrest, advice/mediation, or separation (suspect ordered away for eight hours). The notebook CSV retains only assigned treatment (`assigned`) and delivered treatment (`delivered`), written by code/python/ch3/prepare_data.py; it omits the study's recidivism outcomes.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.